# Predicting Smartphone Addiction — Playground S6E8

**Mission Kaggle #1** · Classification binaire tabulaire · Métrique **ROC AUC**

Approche : features brutes (le feature engineering a été **rejeté** — gain négatif sur ce data synthétique), 5-fold CV stratifié, **blend XGBoost + LightGBM**, poids XGB=0.72 trouvé sur OOF.

Score OOB local : **0.96460**.

In [ ]:
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
N_FOLDS = 5
BLEND_W_XGB = 0.72

train = pd.read_csv('/kaggle/input/playground-series-s6e8/train.csv')
test  = pd.read_csv('/kaggle/input/playground-series-s6e8/test.csv')
sub   = pd.read_csv('/kaggle/input/playground-series-s6e8/sample_submission.csv')
print('train', train.shape, '| test', test.shape)

In [ ]:
NUM = ['age','daily_screen_time_hours','social_media_hours','gaming_hours','work_study_hours',
       'sleep_hours','notifications_per_day','app_opens_per_day','weekend_screen_time']
CAT = ['gender','stress_level','academic_work_impact']

X = train[NUM+CAT].copy(); y = train['addicted_label'].values
Xtest = test[NUM+CAT].copy()
for c in CAT:
    X[c] = X[c].astype('category')
    Xtest[c] = Xtest[c].astype('category')
print('features:', X.shape[1])

In [ ]:
import xgboost as xgb
import lightgbm as lgb

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_xgb = np.zeros(len(X)); oof_lgb = np.zeros(len(X))
test_xgb = np.zeros(len(Xtest)); test_lgb = np.zeros(len(Xtest))

xgb_params = dict(learning_rate=0.05, n_estimators=3000, max_depth=6, min_child_weight=5,
                  tree_method='hist', enable_categorical=True, random_state=RANDOM_STATE, n_jobs=-1)
lgb_params = dict(learning_rate=0.05, n_estimators=3000, num_leaves=31, min_child_samples=20,
                  random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)

for fold, (tr, va) in enumerate(skf.split(X, y)):
    mx = xgb.XGBClassifier(**xgb_params).fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])], verbose=False)
    oof_xgb[va] = mx.predict_proba(X.iloc[va])[:,1]
    test_xgb += mx.predict_proba(Xtest)[:,1]/N_FOLDS

    ml = lgb.LGBMClassifier(**lgb_params).fit(X.iloc[tr], y[tr], categorical_feature=CAT,
        eval_set=[(X.iloc[va], y[va])], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof_lgb[va] = ml.predict_proba(X.iloc[va])[:,1]
    test_lgb += ml.predict_proba(Xtest)[:,1]/N_FOLDS
    print(f'fold {fold+1}: XGB={roc_auc_score(y[va],oof_xgb[va]):.5f} LGB={roc_auc_score(y[va],oof_lgb[va]):.5f}')

In [ ]:
oof_blend = BLEND_W_XGB*oof_xgb + (1-BLEND_W_XGB)*oof_lgb
print('OOB AUC blend:', round(roc_auc_score(y, oof_blend), 5))

In [ ]:
test_blend = BLEND_W_XGB*test_xgb + (1-BLEND_W_XGB)*test_lgb
sub['addicted_label'] = test_blend
sub.to_csv('submission.csv', index=False)
sub.head()